In [1]:
import sys
import os
notebook_dir = os.path.dirname(os.path.abspath(''))
sys.path.insert(0, os.path.abspath(os.path.join(notebook_dir, '..', 'lime_ndt')))
sys.path.insert(0, os.path.abspath(os.path.join(notebook_dir, '..')))

In [16]:
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

# LIME classique (uniquement pour LinearRegression)
from lime.lime_tabular import LimeTabularExplainer as LimeClassicExplainer
# LIME-NDT (gère DecisionTree et NDT)
from lime_ndt.lime_tabular import LimeTabularExplainer as LimeNDTExplainer
from lime_ndt.utils.ndt_sklearn_wrapper import NDTRegressorWrapper

# ========================
# Wrapper pour DecisionTree
# ========================
class DecisionTreeWrapper(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None, *args, **kwargs):
        super().fit(X, y, sample_weight=sample_weight, *args, **kwargs)
        self.coef_ = np.array(self.feature_importances_)
        self.intercept_ = 0
        return self

# ========================
# Charger dataset
# ========================
data = fetch_california_housing()
X = data.data
y = data.target
feature_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

# ========================
# Modèle global (Random Forest)
# ========================
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)

def predict_fn(X):
    return rf.predict(X)

# ========================
# Explainers
# ========================
explainer_classic = LimeClassicExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode='regression'
)

explainer_ndt = LimeNDTExplainer(
    X_train,
    feature_names=feature_names,
    discretize_continuous=False,
    mode='regression'
)

# ========================
# Fonction pour mesurer la fidélité avec train/test de perturbations
# ========================
def fidelity_local_model(instance, explainer, local_model, predict_fn,
                         num_samples_train=5000, num_samples_test=2000, metric="r2"):
    """
    Calcule la fidélité d'un modèle local par rapport au modèle global
    sur des perturbations différentes pour l'entraînement et le test.
    """
    num_features = instance.shape[0]

    # --- Perturbations pour entraîner le modèle local ---
    Z_train = np.zeros((num_samples_train, num_features))
    for i in range(num_features):
        mean = instance[i]
        std = explainer.scaler.scale_[i] if hasattr(explainer.scaler, "scale_") else 0.01
        Z_train[:, i] = np.random.normal(mean, std, size=num_samples_train)
    y_global_train = predict_fn(Z_train)
    local_model.fit(Z_train, y_global_train)

    # --- Perturbations pour tester la fidélité (nouvelles, non vues) ---
    Z_test = np.zeros((num_samples_test, num_features))
    for i in range(num_features):
        mean = instance[i]
        std = explainer.scaler.scale_[i] if hasattr(explainer.scaler, "scale_") else 0.01
        Z_test[:, i] = np.random.normal(mean, std, size=num_samples_test)
    y_global_test = predict_fn(Z_test)
    y_local_test = local_model.predict(Z_test)

    # Calcul de la fidélité
    if metric == "r2":
        return r2_score(y_global_test, y_local_test)
    elif metric == "mse":
        return mean_squared_error(y_global_test, y_local_test)
    else:
        raise ValueError("metric doit être 'r2' ou 'mse'")

# ========================
# Comparer les 3 modèles sur plusieurs instances du test
# ========================
n_instances = 1
results = {"LinearRegression": [], "DecisionTree": [], "NDT": []}

for idx in range(n_instances):
    instance = X_test[idx]
    results["LinearRegression"].append(
        fidelity_local_model(instance, explainer_classic, LinearRegression(), predict_fn)
    )
    results["DecisionTree"].append(
        fidelity_local_model(instance, explainer_ndt, DecisionTreeWrapper(), predict_fn)
    )
    results["NDT"].append(
        fidelity_local_model(instance, explainer_ndt,
                             NDTRegressorWrapper(D=X_train.shape[1], epochs=10, gammas=[100,1]),
                             predict_fn)
    )

# Moyenne sur toutes les instances
print("=== Fidélité moyenne des modèles locaux (R²) ===")
for model_name, scores in results.items():
    print(f"{model_name}: {np.mean(scores):.3f}")


mean_leaf_values shape: (32, 1, 1)
self.L: 32 self.C: 1
mean_leaf_values shape after squeeze: (32, 1)


c:\Users\DELL\Desktop\lime_ndt\.venv\lib\site-packages\keras\src\optimizers\base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
=== Fidélité moyenne des modèles locaux (R²) ===
LinearRegression: 0.435
DecisionTree: 0.837
NDT: 0.044
